In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/arXiv_scientific_dataset.csv")

df = df[[
    "title",
    "category",
    "summary",
    "category_code",
    "authors",
    "first_author",
    "published_date",
]]

df.rename(columns={
    "title": "titles",
    "summary": "summaries",
    "category_code": "terms"
}, inplace=True)

df = df.dropna()

print("Initial size:", len(df))

Initial size: 136238


In [ ]:
df["terms"] = df["terms"].apply(lambda x: x.split())

from collections import Counter

all_labels = df["terms"].explode()
label_counts = Counter(all_labels)

min_count = 15
valid_labels = {l for l, c in label_counts.items() if c >= min_count}

df["terms"] = df["terms"].apply(lambda x: [l for l in x if l in valid_labels])
df = df[df["terms"].map(len) > 0]

print("After label filtering:", len(df))

df.to_csv("../data/processed_dataset.csv", index=False)

After label filtering: 135939


In [3]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)
val_df = test_df.sample(frac=0.5, random_state=42)
test_df = test_df.drop(val_df.index)

print(len(train_df), len(val_df), len(test_df))

122345 6797 6797


In [4]:
# Later version
train_df["text"] = train_df["titles"] + " " + train_df["summaries"]
val_df["text"] = val_df["titles"] + " " + val_df["summaries"]
test_df["text"] = test_df["titles"] + " " + test_df["summaries"]

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=80000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.9,
    stop_words="english",
    sublinear_tf=True
)

X_train = vectorizer.fit_transform(train_df["text"])
X_val = vectorizer.transform(val_df["text"])
X_test = vectorizer.transform(test_df["text"])

In [6]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

y_train = mlb.fit_transform(train_df["terms"])
y_val = mlb.transform(val_df["terms"])
y_test = mlb.transform(test_df["terms"])

print("Total labels:", len(mlb.classes_))

Total labels: 80


In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier

# Logistic Regression
model_lr = OneVsRestClassifier(
    LogisticRegression(max_iter=2000, class_weight="balanced", n_jobs=-1)
)
model_lr.fit(X_train, y_train)

# Naive Bayes
model_nb = OneVsRestClassifier(MultinomialNB())
model_nb.fit(X_train, y_train)

# 🔥 SVM (BIG BOOST)
model_svm = OneVsRestClassifier(LinearSVC())
model_svm.fit(X_train, y_train)

,estimator,LinearSVC()
,n_jobs,None
,verbose,0
,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1


In [8]:
# Ensemble

probs_lr = model_lr.predict_proba(X_test)
probs_nb = model_nb.predict_proba(X_test)

# SVM gives decision scores (not probabilities)
probs_svm = model_svm.decision_function(X_test)

# normalize SVM scores to 0-1
probs_svm = (probs_svm - probs_svm.min()) / (probs_svm.max() - probs_svm.min())

# weighted ensemble
final_probs = (
    0.5 * probs_lr +
    0.3 * probs_nb +
    0.2 * probs_svm
)

# sharpening
final_probs = np.power(final_probs, 1.2)

In [9]:
def predict_dynamic_k(probs, threshold=0.3, max_k=4, min_k=2):
    y_pred = np.zeros_like(probs)
    
    for i in range(probs.shape[0]):
        idx = np.where(probs[i] >= threshold)[0]
        
        if len(idx) == 0:
            idx = np.argsort(probs[i])[-min_k:]
        elif len(idx) > max_k:
            idx = np.argsort(probs[i])[-max_k:]
            
        y_pred[i, idx] = 1
        
    return y_pred

# y_pred = predict_dynamic_k(final_probs)

# # def predict_top_k_matrix(probs, k=1):
# #     y_pred = np.zeros_like(probs)
    
# #     for i in range(probs.shape[0]):
# #         top_idx = np.argsort(probs[i])[-k:]
# #         y_pred[i, top_idx] = 1
        
# #     return y_pred

# # k = 1
# # y_pred = predict_top_k_matrix(final_probs, k=k)

# def predict_dynamic_k(probs, threshold=0.3, max_k=2):
#     y_pred = np.zeros_like(probs)
    
#     for i in range(probs.shape[0]):
#         idx = np.where(probs[i] >= threshold)[0]
        
#         if len(idx) == 0:
#             idx = np.argsort(probs[i])[-1:]
#         elif len(idx) > max_k:
#             idx = np.argsort(probs[i])[-max_k:]
            
#         y_pred[i, idx] = 1
        
#     return y_pred

# y_pred = predict_dynamic_k(final_probs, threshold=0.3, max_k=2)

In [10]:
# Threshold tunning

from sklearn.metrics import f1_score

best_thresh = 0
best_f1 = 0

for t in np.arange(0.2, 0.6, 0.05):
    y_pred_temp = predict_dynamic_k(final_probs, threshold=t, max_k=3)
    f1 = f1_score(y_test, y_pred_temp, average="macro")
    
    print(f"Threshold {t:.2f} → F1: {f1:.4f}")
    
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

print("\n🔥 Best Threshold:", best_thresh)

C:\Users\Harshhal\OneDrive\Desktop\Python\tf-env\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Harshhal\OneDrive\Desktop\Python\tf-env\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Threshold 0.20 → F1: 0.2747
Threshold 0.25 → F1: 0.2873


C:\Users\Harshhal\OneDrive\Desktop\Python\tf-env\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Harshhal\OneDrive\Desktop\Python\tf-env\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Harshhal\OneDrive\Desktop\Python\tf-env\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(

Threshold 0.30 → F1: 0.2930
Threshold 0.35 → F1: 0.2957
Threshold 0.40 → F1: 0.2917


C:\Users\Harshhal\OneDrive\Desktop\Python\tf-env\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Harshhal\OneDrive\Desktop\Python\tf-env\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Threshold 0.45 → F1: 0.2860
Threshold 0.50 → F1: 0.2756
Threshold 0.55 → F1: 0.2444

🔥 Best Threshold: 0.35


C:\Users\Harshhal\OneDrive\Desktop\Python\tf-env\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [11]:
# from sklearn.metrics import f1_score, precision_score, recall_score

# print("F1 micro:", f1_score(y_test, y_pred, average="micro"))
# print("F1 macro:", f1_score(y_test, y_pred, average="macro"))
# print("Precision:", precision_score(y_test, y_pred, average="micro"))
# print("Recall:", recall_score(y_test, y_pred, average="micro"))


from sklearn.metrics import precision_score, recall_score

y_pred = predict_dynamic_k(final_probs, threshold=best_thresh, max_k=3)

print("F1 micro:", f1_score(y_test, y_pred, average="micro"))
print("F1 macro:", f1_score(y_test, y_pred, average="macro"))
print("Precision:", precision_score(y_test, y_pred, average="micro"))
print("Recall:", recall_score(y_test, y_pred, average="micro"))

print("Avg labels per sample:", y_pred.sum(axis=1).mean())

F1 micro: 0.7138607249104487
F1 macro: 0.2957421567908693


C:\Users\Harshhal\OneDrive\Desktop\Python\tf-env\lib\site-packages\sklearn\metrics\_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Precision: 0.6077113913582799
Recall: 0.8649404148889216
Avg labels per sample: 1.423274974253347


In [12]:
from sentence_transformers import SentenceTransformer
import pickle

df["text"] = df["titles"] + " " + df["summaries"]

model_st = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model_st.encode(
    df["text"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Embeddings shape:", embeddings.shape)

C:\Users\Harshhal\OneDrive\Desktop\Python\tf-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Harshhal\OneDrive\Desktop\Python\tf-env\lib\site-packages\google\api_core\_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Batches: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 2125/2125 [1:38:42<00:00,  2.79s/it]


Embeddings shape: (135939, 384)


In [14]:
import os
import pickle

# create models folder
os.makedirs("../models", exist_ok=True)

# =========================
# 🔹 SAVE CLASSIFICATION MODELS
# =========================
pickle.dump(model_lr, open("../models/model_lr.pkl", "wb"))
pickle.dump(model_nb, open("../models/model_nb.pkl", "wb"))
pickle.dump(model_svm, open("../models/model_svm.pkl", "wb"))

# =========================
# 🔹 SAVE VECTORIZER + LABEL BINARIZER
# =========================
pickle.dump(vectorizer, open("../models/vectorizer.pkl", "wb"))
pickle.dump(mlb, open("../models/mlb.pkl", "wb"))

# =========================
# 🔹 SAVE THRESHOLD
# =========================
pickle.dump(best_thresh, open("../models/threshold.pkl", "wb"))

# =========================
# 🔹 SAVE EMBEDDINGS (VERY IMPORTANT)
# =========================
pickle.dump(embeddings, open("../models/embeddings.pkl", "wb"))

# =========================
# 🔹 SAVE SENTENCE TRANSFORMER MODEL NAME
# =========================
with open("../models/embedding_model.txt", "w") as f:
    f.write("all-MiniLM-L6-v2")

print("✅ All models and embeddings saved successfully!")

✅ All models and embeddings saved successfully!
